In [1]:
import numpy as np
from numba import njit

from gtsimulation.Global import Units
from gtsimulation.MagneticFields import AbsBfield
from gtsimulation.Particle.Generators import Distributions, Spectrums
from gtsimulation.Particle import Flux

from scipy.integrate import quad
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import interp1d
from scipy.interpolate import RegularGridInterpolator
from scipy.special import kv
import matplotlib.pyplot as plt

## Предварительные вычисления
Тут вычисляются и сохраняются в файлы таблицы для формул которые очень долго вычислять во время симуляции

### Квазиклассическое приближение вероятности
Тут вычисляется интегральная часть квазиклассического приближения шанса реакции:

$W\left(SQA\right)=\frac{\alpha m^{2}}{3\sqrt{3}\pi\omega}\int_{-1}^{1}\frac{9-v^{2}}{1-v^{2}}K_{2/3}\left(\frac{8}{3\left(1-v^{2}\right)\chi}\right)dv$

,где $K_{2/3}$ - модифицированная функция Бесселя
W или R - это вероятность реакции на ед. пути с размерностью 1/м

In [2]:
def dimensionless_sqa_integral(chi):
    # Интеграл из формулы (3.7) [4]
    integrand = lambda v: ((9 - v**2) / (1 - v**2)) * kv(2/3, 8 / (3 * (1 - v**2) * chi))
    res, _ = quad(integrand, -1, 1)
    return res

chi_grid = np.logspace(-2, 2, 500) 
table_values = np.array([dimensionless_sqa_integral(c) for c in chi_grid])
np.savez('sqa_lookup_table.npz', chi=chi_grid, values=table_values)

### Квазиклассическое приближение рапределения энергии
Тут вычисляется распределение энергии между элекртроном и позитроном. Используется метод обратной функции, тоесть подается случайное число и выдается доля энергии.

Так как для разных величин магнитной жесткости, χ, разные распеределения, то создаётся двумерная таблица от χ и случайного числа.

Формула распределения:

$\frac{dP(\epsilon,\chi)}{d\epsilon}=\frac{\alpha m^{2}}{3\sqrt{3}\pi\omega}\ \frac{\left(2+​\epsilon\left(1-\epsilon\right)\right)}{\epsilon\left(1-\epsilon\right)}K_{2/3​}\left(\frac{1}{3\epsilon\chi\left(1-\epsilon\right)}\right)$

In [3]:
prob_grid = np.linspace(0, 1, 1000)
chi_grid = np.logspace(-1, 20, 1000)
epsilon_table = np.zeros((len(chi_grid), len(prob_grid)))
cdf_table = np.zeros((len(chi_grid), len(prob_grid)))

# Сетка для интегрирования (от 0 до 0.5 по симметрии)
e_integration_grid = np.linspace(1e-5, 0.5, 500)

for i, chi in enumerate(chi_grid):
    term_bessel = 1.0 / (3.0 * chi * e_integration_grid * (1.0 - e_integration_grid))
    factor = (2.0 + e_integration_grid * (1.0 - e_integration_grid)) / (e_integration_grid * (1.0 - e_integration_grid))
    pdf_values = factor * kv(2/3, term_bessel)
    
    cdf_values = cumulative_trapezoid(pdf_values, e_integration_grid, initial=0)
    total_sum = cdf_values[-1]

    if total_sum > 1e-100: # используем очень малое число вместо 0
        cdf_values /= total_sum
        
        # берем только те индексы, где CDF растет
        # это убирает плоские участки которые не дают работать interp1d
        _, idx = np.unique(cdf_values, return_index=True)
        
        # для интерполяции нам нужно сохранить исходный порядок сетки по e_integration_grid
        idx = np.sort(idx)

        # проверяем, что уникальных точек хотя бы 2, иначе интерполяция невозможна
        if len(idx) > 1:
            inv_func = interp1d(cdf_values[idx], 
                                e_integration_grid[idx], 
                                kind='linear', fill_value="extrapolate")
            epsilon_table[i, :] = inv_func(prob_grid)
        else:
            # затычка/костыль
            epsilon_table[i, :] = 0.5

if not np.isfinite(epsilon_table).all():
    print("В таблице все еще есть NaN!")

# for row in cdf_table[:100]:
#     print("".join(f"{item==0}" for item in row))

CDF_Interpolator = RegularGridInterpolator(
                points=(np.log10(chi_grid), prob_grid),
                values=epsilon_table,
                method='cubic',
                bounds_error=False,
                fill_value=0.5         # значение по умолчанию при выходе за границы
            )

In [4]:
np.savez_compressed(
    "epsilon_interpolation_data.npz", 
    epsilon_table=epsilon_table,
    chi_grid=chi_grid,
    prob_grid=prob_grid
)

print("Создана и сохранена таблица интерполяции эпсилон")

Создана и сохранена таблица интерполяции эпсилон


## Магнитное поле
Тут я создаю класс магнитного поля пульсара.

Он из себя представляет в приближении диполь с моментом вдоль оси Z.

In [11]:
class dipole_field(AbsBfield):
    def __init__(self, m0: float) -> None:
        self.m0 = m0
        self.m_vector = np.array([0, 0, 1]) * m0
        self.use_meters = True
        self.use_tesla = True


    def CalcBfield(self, x, y, z):
        return np.array(self.__calc_bfield(x, y, z, self.m_vector))
    
    @staticmethod
    @njit(fastmath=True)
    def __calc_bfield(x, y, z, m_vector):
        r_mag2 = x**2 + y**2 + z**2
        r_mag = np.sqrt(r_mag2)
    
        if r_mag == 0.0:
            return 0.0, 0.0, 0.0
        
        dot_product = m_vector[0]*x + m_vector[1]*y + m_vector[2]*z
    
        r3_inv = 1.0 / (r_mag2 * r_mag)       # 1 / r^3
        r5_inv = r3_inv / r_mag2              # 1 / r^5
    
        factor1 = 3.0 * dot_product * r5_inv  
        factor2 = r3_inv                       
    
        Bx = 1e-7 * (factor1 * x - factor2 * m_vector[0])
        By = 1e-7 * (factor1 * y - factor2 * m_vector[1])
        Bz = 1e-7 * (factor1 * z - factor2 * m_vector[2])
    
        return Bx, By, Bz

    def UpdateState(self, newDate):
        pass

    def to_string(self):
        s = f"""Type: Dipole Magnetic Field
        Version: 1.0"""

        return s

## Симулятор
Я взял BorisBuneman и GTSimulator скопировал их код и приписал свои функции, потому что нельзя было наследовать приватные функции

Также симулятор в отдельном файле для удобства

В него я добавил: инициализацию конверсии фотонов `__SetPMConversion`, главную функцию  `__PMConversion`, функцию вычисляющию энергии частиц `__get_electrons_energies` и их моментов `__sample_pair_momenta`.

In [ ]:
from PhotonSimulator import PhotonSimulator

### Переменные
Тут задаю переменные симуляции

Момент магнитного поля такой большой чтобы был большой шанс что реакция произойдет

In [6]:
dt = 1e-9
n_steps = 1e3

In [7]:
m0 = 0.5e50 #амплитуда момента магнитного поля
border = 1000000
R_star = 10000
step = {"UseAdaptiveStep": True, "InitialStep": 1}
radLosses = [True, {"Photons": True, "MinE": Units.GeV, "MaxE": 1e20}]

In [8]:
T = 1000000 * Units.MeV
latitude = 0
r_0 = np.array([10000, 0, 0]) 
v_0 = np.array([np.cos(latitude), 0, np.sin(latitude)])

ph = Flux(
    Spectrum=Spectrums.Monolines(energy=T),
    Distribution=Distributions.UserInput(
        R0=r_0,
        V0=v_0
    ),
    PDGcode=22,
    Nevents=1
)

b_field = dipole_field(m0)

sim = PhotonSimulator(
    Bfield=b_field,
    Particles=ph,
    Step=dt,
    Num=n_steps,
    Save=[1, {"Coordinates": True, "Velocities": True, "Energy": True}],
    Verbose=2,
    PMConversion=True,
)

Creating simulator object...
Date: 2008-01-01 00:00:00
Time step: 1e-09 seconds
Number of steps: 1000
Radiation Losses: False
Synchrotron Emission: False
Region: Undefined
		Additional Energy Losses: False
Number of files: 1
Output file name: None_num.npy
Save every 1 step of:
	Coordinates: True
	Velocities: True
	Efield: False
	Bfield: False
	Angles: False
	Path: False
	Density: False
	Clock: False
	Energy: True
	PitchAngles: False
	LarmorRadii: False
	GuidingCenter: False
Electric field: None
Magnetic field: Type: Dipole Magnetic Field
        Version: 1.0
Medium: None
Decay: False
Nuclear Interactions: None
Flux: 
        Number of particles: 1
        V: Isotropic
        Spectrum: Monolines
        Energy: 1000000
        Distribution: User Input
        R0 shape: (1, 3)
        V0 shape: (1, 3)
Tracing: Inward
Break Conditions:
	Xmin: 0.0
	Ymin: 0.0
	Zmin: 0.0
	Rmin: 0.0
	Dist2Path: 0.0
	Xmax: inf
	Ymax: inf
	Zmax: inf
	Rmax: inf
	MaxPath: inf
	MaxTime: inf
	MaxRev: inf
	Death: i

In [9]:
track = sim()[0][0]

Launching simulation...

File 1/1 started

Event 1/1 started
Particle: gamma (M = 0.000000 [MeV/c2], Z = 0)
Energy: 1000000.000000 [MeV], Rigidity: inf [GV]
Coordinates: [10000.     0.     0.] [m]
Velocity: [1. 0. 0.]
Beta: 1.0
Beta * dt: 29.979246 [m]
Calculating:
Pair Production
Epsilon : 0.017290952910223163
### Break due to Death ###
	Progress: 100%
Event 1/1 finished in 0.838 seconds

Simulation completed!


In [10]:
r = track["Track"]["Coordinates"]
ch = track["Child"]
    
print(ch)

[{'Track': {'Coordinates': array([10029.9792458,     0.       ,     0.       ]), 'Velocities': array([1., 0., 0.])}, 'Particle': {'PDG': 11, 'M': 0.5109989461, 'T': np.float64(17290.44191127706), 'Gen': 1}}, {'Track': {'Coordinates': array([10029.9792458,     0.       ,     0.       ]), 'Velocities': array([1., 0., 0.])}, 'Particle': {'PDG': -11, 'M': 0.5109989461, 'T': np.float64(982708.5360908307), 'Gen': 1}}]


Видно что направление скорости совпадает с направением фотона, а энергия почти вся перешла 1ой частице, что я и ожидал